# GRU — Scheme 2 (Static Test) — GAMEEMO

> Run on **Google Colab**. Mount your Google Drive and adjust `folder_path` before executing.


In [ ]:
import os, numpy as np, pandas as pd
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ── Define training size (change per experiment step) ──
train_size = 0.95  # fraction of training pool used at this step

folder_path   = '/content/drive/My Drive/EEG Datasets/GAMEEMO/Denoised EEG Data/'
test_file_path = folder_path + 'gameemo_test.csv'

subject_files = [[f"S{i:02}_G{j}_Denoised.csv" for j in range(1, 5)] for i in range(1, 29)]
data_list = []
for subject_file_list in subject_files:
    for file in subject_file_list:
        temp_data = pd.read_csv(os.path.join(folder_path, file))
        data_list.append(temp_data)

train_pool = pd.concat(data_list, ignore_index=True)
train_pool = train_pool.sample(frac=0.10, random_state=42).reset_index(drop=True)

# ── Load the FIXED static test set (never subsampled) ──
test_data = pd.read_csv(test_file_path)

# ── Subsample training pool ──
data_train = train_pool.sample(frac=train_size, random_state=42).reset_index(drop=True)

X_train_raw = data_train.drop(columns=['Valence', 'Arousal']).values
y_train_raw = data_train['Valence'].values
X_test_raw  = test_data.drop(columns=['Valence', 'Arousal']).values
y_test_raw  = test_data['Valence'].values

# ── Fit scaler and encoder on TRAINING data ONLY ──
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

valence_enc = LabelEncoder()
y_train = valence_enc.fit_transform(y_train_raw)
y_test  = valence_enc.transform(y_test_raw)

# ────────────────────────────────────────────────────────────
# GRU — Scheme 2 (Static Test) — GAMEEMO
# ────────────────────────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Dropout, Dense

X_train_in = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_in  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)
X_cv = X_train_in

def build_model():
    m = Sequential([
        Input(shape=(X_train_in.shape[1], 1)),
        GRU(64, return_sequences=True), Dropout(0.2),
        GRU(32), Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

num_classes = len(np.unique(y_train))

# ── 5-Fold Cross-Validation ──
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_accuracies = []
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_cv), 1):
    m = build_model()
    m.fit(X_cv[tr_idx], y_train[tr_idx], epochs=10, batch_size=32, verbose=0)
    _, acc = m.evaluate(X_cv[val_idx], y_train[val_idx], verbose=0)
    cv_accuracies.append(acc)
    print(f"Fold {fold} Accuracy: {acc:.4f}")
print(f"5-Fold CV Accuracy with {int(train_size * 100)}% training data: {np.mean(cv_accuracies):.4f} ± {np.std(cv_accuracies):.4f} (SD)")

final_model = build_model()
final_model.fit(X_train_in, y_train, epochs=10, batch_size=32, verbose=1,
                validation_data=(X_test_in, y_test))
_, accuracy = final_model.evaluate(X_test_in, y_test, verbose=0)
print(f"Held-out Test Accuracy with {int(train_size * 100)}% training data: {accuracy:.4f}")

y_pred = np.argmax(final_model.predict(X_test_in), axis=1)
cm = confusion_matrix(y_test, y_pred, normalize='true')
plt.figure(figsize=(5, 4))
sns.heatmap(cm * 100, annot=True, fmt=".2f", cmap="Blues")
plt.title(f"Confusion Matrix ({int(train_size * 100)}% Training Data)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()